<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

<br>
<h1 style="font-family:verdana; font-size:26px"> <center>~ Notebook Penyisihan Objective Quest Dataquest Universitas Airlannga 2025 ~</center> </h1>
<br>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# **InfoMagangDong**

<p style = "font-size:16px; font-family:verdana">- Mohammad Maulana Firdaus Ramadhan (Leader)<p>

<p style = "font-size:16px; font-family:verdana">- Muhammad Fathur Rizky <p>

<p style = "font-size:16px; font-family:verdana">- Razi Rachman Widyadhana <p>

<br>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Daftar Isi

1. [**Introduction**](#1)

2. [**Preprocessing**](#2)

3. [**Modelling**](#3)

4. [**Submission**](#4)

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Introduction <a name="1"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Permasalahan

Lorem Ipsum

---

## Environment Setup

In [ ]:
!nvidia-smi

Sun Sep  7 08:03:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000               On  |   00000000:03:00.0 Off |                  Off |
| 37%   55C    P8             24W /  300W |       0MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128
!pip uninstall keras -y
!pip install tf-keras
!pip uninstall transformers -y
!pip install transformers[torch]
!pip install xgboost catboost lightgbm

Looking in indexes: https://download.pytorch.org/whl/cu128, https://pypi.ngc.nvidia.com
Found existing installation: keras 3.11.3
Uninstalling keras-3.11.3:
  Successfully uninstalled keras-3.11.3
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 162.6 MB/s eta 0:00:00
Found existing installation: transformers 4.56.1
Uninstalling transformers-4.56.1:
  Successfully uninstalled transformers-4.56.1
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 220.6 MB/s eta 0:00:00
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


---

## Pustaka

In [ ]:
import os
import re
import gc
import random
import numpy as np
import pandas as pd

import glob
import zipfile
from tqdm import tqdm
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

2025-09-07 08:04:04.449880: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
print(torch.cuda.is_available())

# print(torch.cuda.get_device_name(0))

True


---

## Directory Setup

**NOTE:** Pastikan peletakan ZIP file pada path `./data/objective-quest-2025.zip`

```latex
├── your-project                           # main project root
│   │
│   ├── data                   
│   │   │
│   │   └── objective-quest-2025.zip
│   │
│   └── Notebook_InfoMagangDong.ipynb
```

In [ ]:
SUBMISSION_DIR = Path("./submissions")

if not SUBMISSION_DIR.exists():
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_zip(zip_path, extract_to=None):

    if extract_to is None:
        extract_to = os.path.dirname(zip_path)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"Successfully extracted {zip_path} to {extract_to}")

    except FileNotFoundError:
        print(f"Error: Zip file not found at {zip_path}")
    except zipfile.BadZipFile:
        print(f"Error: Invalid zip file at {zip_path}")
    except Exception as e:
        print(f"Error extracting zip file: {e}")

# extract_zip("data/objective-quest-2025.zip")
print(f"Successfully extracted data/objective-quest-2025.zip to data")

Successfully extracted data/objective-quest-2025.zip to data


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Preprocessing <a name="2"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

In [ ]:
DEBUG = False

class CourtProcessor:
    def __init__(self):

        cleaning_patterns = [
            r'\bhal(?:aman)?\s+\d+(?:\s+dari\s+\d+(?:\s+hal(?:aman)?)?)?',                                                                                     # page numbers
            r'direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia',                                                                                  # directory header
            r'\bmah?k?amah?\s+agung\s+republ(?:ik)?\s+indonesi(?:a)?\b',                                                                                       # mahkamah agung
            r'kepaniteraan\s+mahkamah\s+agung\s+ri',                                                                                                           # kepaniteraan
            r'\b(?:hkama|ahkamah agung repub|ahkamah agung republik indonesia|mah agung republik indonesia|blik indonesi)\b',                                  # OCR fragments
            r'.*(?:transparansi dan akuntabilitas|fungsi peradilan|inakurasi informasi|keterkinian informasi|informasi yang kami sajikan|disclaimer|email).*'  # disclaimer
        ]
        self.combined_cleaning_pattern = re.compile('|'.join(f'({pattern})' for pattern in cleaning_patterns))

        # ---    Verdict section patterns ---
        self.start_pattern = re.compile(r"(?:setelah|telah)\s+mendengar(?:kan)?\s+(?:uraian\s+)?(?:pembacaan\s+)?(?:surat\s+)?tuntutan\s+(?:pidana|hukum)?")
        self.alt_start_pattern = re.compile(r"(?:(?:(?:1|2|\ba|\bb)[\.\)]*|\.|\)|\-|•|memutuskan:)[\s\n]*(?:m\s*e\s*n\s*y\s*a\s*t\s*a\s*k\s*a\s*n|menyatakan.*bersalah|melakukan kekerasan))")
        self.final_start_pattern = re.compile(r"(?:(yang memeriksa dan mengadili perkara ini (?:memutuskan|memberikan)[:\s\n]))")
        self.end_pattern = re.compile(r"menyatakan barang bukti|barang bukti")
        self.alt_end_pattern = re.compile(r"(?:setelah|telah)\s+mendengar|menimbang, bahwa|menimbang bahwa")

        # --- Final cleaning patterns ---
        self.final_cleaning_pattern = re.compile(r'([.\-])\1+|[;/:,"\'`_…]')
        self.unicode_pattern = re.compile(r'[^\x00-\x7F]+')
        self.spaces_pattern = re.compile(r'\s+')

        # --- Lifetime keywords pattern ---
        self.lifetime_pattern = re.compile(r'\b(?:seumur|pidana mati|seumur hidup)\b')

    def read_file_content(self, file_path):

        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                return file.read()
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
            return None

    def initial_cleaning(self, text):

        text = text.lower()
        text = self.combined_cleaning_pattern.sub('', text)
        text = re.sub(r'menayatakan|menyatkan|menyataan|\(menyatakan|--menyatakaan|menyatakac', 'menyatakan', text)
        text = self.spaces_pattern.sub(' ', text).strip()

        return text

    def extract_verdict_section(self, text):

        if DEBUG:
            print("Cleaned text preview:")
            print(text[:500])

        start_match = self.start_pattern.search(text)

        if not start_match:

            if DEBUG:
                print("No start match with primary pattern, trying alternative.")

            start_match = self.alt_start_pattern.search(text)

        if not start_match:

            if DEBUG:
                print("No start match with alternative pattern, trying final.")

            start_match = self.final_start_pattern.search(text)

        if DEBUG:
            print("Start match:", start_match)

        if not start_match:
            return None

        start_idx = start_match.end()
        end_match = self.end_pattern.search(text, start_idx)
        alt_end_match = self.alt_end_pattern.search(text, start_idx)

        end_positions = []
        if end_match:
            end_positions.append(end_match.start())
        if alt_end_match:
            end_positions.append(alt_end_match.start())

        end_idx = min(end_positions) if end_positions else len(text)

        if DEBUG:
            print("End match:", end_match)
            print("Alt end match:", alt_end_match)
            print("Selected end_idx:", end_idx)

        extracted_text = text[start_idx:end_idx].strip()

        if DEBUG:
            print("Extracted text:", extracted_text)

        return extracted_text if extracted_text else None

    def check_lifetime_verdict(self, text):

        if text:
            return bool(self.lifetime_pattern.search(text))

        return False

    def final_cleaning(self, text):

        text = self.final_cleaning_pattern.sub(' ', text)
        text = self.unicode_pattern.sub(' ', text)

        # --- Remove numbered list items (e.g., "1.", "2.", "3.") when surrounded by spaces or at boundaries ---
        text = re.sub(r'(?:^|\s)\d+\.(?=\s|$)', ' ', text)

        text = re.sub(r"yang diajukan oleh penuntut umum yang pada pokoknya sebagai berikut", " ", text)
        text = re.sub(r"menyatakan terdakwa", " ", text)
        text = re.sub(r"dari penuntut umum yang pada pokoknya sebagai berikut", " ", text)
        text = re.sub(r"yang diajukan oleh penuntut umum", " ", text)
        text = re.sub(r"penuntut umum yang pada pokoknya", " ", text)
        text = re.sub(r"sebagai berikut", " ", text)
        text = re.sub(r"m e n u n t u t", " ", text)
        text = re.sub(r"dari menuntut", " ", text)
        text = re.sub(r"penuntut umum", " ", text)
        text = re.sub(r"yang pada pokoknya menuntut", " ", text)
        text = re.sub(r"p u t u s a n nomor", " ", text)
        text = re.sub(r"p e t i k a n", " ", text)
        text = re.sub(r"c a t a t a n p e r s i d a n g a n", " ", text)

        text = self.spaces_pattern.sub(' ', text).strip()
        return text

    def save_results_to_csv(self, results, columns, output_filename, project_root, description):

        if results:
            df = pd.DataFrame(results, columns=columns)

            # --- Filter out rows with 88888 in 'lama hukuman (bulan)' column if it exists ---
            if 'lama hukuman (bulan)' in df.columns:
                original_count = len(df)
                df = df[df['lama hukuman (bulan)'] != 88888]
                filtered_count = original_count - len(df)
                if filtered_count > 0:
                    print(f"Filtered out {filtered_count} records with lama hukuman = 88888")

            # --- Filter out rows with empty text ---
            if 'text' in df.columns:
                original_count = len(df)
                df = df[df['text'].str.strip() != '']
                filtered_count = original_count - len(df)
                if filtered_count > 0:
                    print(f"Filtered out {filtered_count} records with empty text")

            output_path = os.path.join(project_root, "preprocessed_data", output_filename)
            # Create directory if it doesn't exist
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            df.to_csv(output_path, index=False)
            print(f"Saved {description} to {output_path}")
            print(f"{description.capitalize()} shape: {df.shape}")

    def process_single_file(self, file_path):

        text = self.read_file_content(file_path)
        if not text:
            return "", False, False

        cleaned_text = self.initial_cleaning(text)
        verdict_section = self.extract_verdict_section(cleaned_text)

        is_standard = (verdict_section is not None and
                      verdict_section.strip() != "" and
                      any(keyword in verdict_section for keyword in ["selama", "menjatuhkan", "menghukum", "seumur hidup"]))

        final_text = verdict_section if is_standard else cleaned_text

        lifetime = False
        if is_standard and verdict_section:
            lifetime = self.check_lifetime_verdict(verdict_section)

        if not final_text:
            final_text = ""
        else:
            final_text = self.final_cleaning(final_text)

        return final_text, lifetime, is_standard

    def process_files(self, project_root, text_dir):
        train_path = os.path.join(project_root, "data", "train.csv")
        test_path = os.path.join(project_root, "data", "test.csv")

        train_ids = set()
        test_ids = set()
        train_targets = {}

        if os.path.exists(train_path):
            train_df = pd.read_csv(train_path)
            train_ids = set(train_df['id'].values)
            train_targets = dict(zip(train_df['id'], train_df.get('lama hukuman (bulan)', [None] * len(train_df))))
            print(f"Loaded {len(train_ids)} training document IDs")

        if os.path.exists(test_path):
            test_df = pd.read_csv(test_path)
            test_ids = set(test_df['id'].values)
            print(f"Loaded {len(test_ids)} test document IDs")

        all_ids = train_ids.union(test_ids)

        file_pattern = os.path.join(project_root, text_dir, "*.txt")
        txt_files = glob.glob(file_pattern)

        if not txt_files:
            print(f"No .txt files found in {text_dir}")
            return

        # --- Filter files to only process those in train/test sets ---
        relevant_files = []
        for file_path in txt_files:
            file_name = os.path.basename(file_path)
            doc_id = os.path.splitext(file_name)[0]
            if doc_id in all_ids:
                relevant_files.append((file_path, doc_id))

        print(f"Processing {len(relevant_files)} files (out of {len(txt_files)} total) that match train/test IDs")

        # --- Process all files in a single loop ---
        train_standard_results = []
        test_standard_results = []
        train_different_results = []
        test_different_results = []
        train_lifetime_results = []
        test_lifetime_results = []

        for file_path, doc_id in tqdm(relevant_files, desc="Processing documents", unit="file"):

            final_text, lifetime, is_standard = self.process_single_file(file_path)

            if is_standard and len(final_text) > 2000:
                final_text = final_text[:2000]

                if DEBUG:
                    tqdm.write(f"📏 {doc_id}: Text truncated to 2000 characters")

            if lifetime and DEBUG:
                tqdm.write(f"! {doc_id}: Lifetime verdict found")

            if not is_standard and DEBUG:
                tqdm.write(f"- {doc_id}: Different convention detected")

            if not final_text and DEBUG:
                tqdm.write(f"✗ {doc_id}: No text content found")

            result = {
                'id': doc_id,
                'text': final_text
            }

            # --- Add to appropriate dataset ---
            if doc_id in train_ids:
                result['lama hukuman (bulan)'] = train_targets.get(doc_id, None)
                if lifetime:
                    train_lifetime_results.append(result)
                elif is_standard:
                    train_standard_results.append(result)
                else:
                    train_different_results.append(result)
            else:
                if lifetime:
                    test_lifetime_results.append(result)
                elif is_standard:
                    test_standard_results.append(result)
                else:
                    test_different_results.append(result)

        # --- Save results for standard files ---
        self.save_results_to_csv(
            train_standard_results,
            ['id', 'text', 'lama hukuman (bulan)'],
            "train_standard.csv",
            project_root,
            "standard training data"
        )

        self.save_results_to_csv(
            test_standard_results,
            ['id', 'text'],
            "test_standard.csv",
            project_root,
            "standard test data"
        )

        # --- Save results for anomaly files ---
        self.save_results_to_csv(
            train_different_results,
            ['id', 'text', 'lama hukuman (bulan)'],
            "train_anomaly.csv",
            project_root,
            "different training data"
        )

        self.save_results_to_csv(
            test_different_results,
            ['id', 'text'],
            "test_anomaly.csv",
            project_root,
            "different test data"
        )

        # --- Save lifetime verdict files ---
        self.save_results_to_csv(
            train_lifetime_results,
            ['id', 'text', 'lama hukuman (bulan)'],
            "train_lifetime.csv",
            project_root,
            "training lifetime verdicts"
        )

        self.save_results_to_csv(
            test_lifetime_results,
            ['id', 'text'],
            "test_lifetime.csv",
            project_root,
            "test lifetime verdicts"
        )

        # --- Print summary statistics ---
        total_standard = len(train_standard_results) + len(test_standard_results)
        total_different = len(train_different_results) + len(test_different_results)
        total_files = total_standard + total_different
        total_lifetime = len(train_lifetime_results) + len(test_lifetime_results)

        print(f"\nSummary:")
        print(f"Total files processed: {total_files}")
        print(f"Standard convention files: {total_standard} ({total_standard/total_files*100:.1f}%)")
        print(f"Different convention files: {total_different} ({total_different/total_files*100:.1f}%)")
        print(f"Lifetime verdicts found: {total_lifetime} ({total_lifetime/total_files*100:.1f}%)")
        print(f"Dataset breakdown:")
        print(f"  Standard train files: {len(train_standard_results)}")
        print(f"  Standard test files: {len(test_standard_results)}")
        print(f"  Different train files: {len(train_different_results)}")
        print(f"  Different test files: {len(test_different_results)}")
        print(f"Lifetime sentence breakdown:")
        print(f"  Train lifetime sentences: {len(train_lifetime_results)}")
        print(f"  Test lifetime sentences: {len(test_lifetime_results)}")

    def run(self, text_dir, project_root=None):
        # Fix for Jupyter notebook environment
        if project_root is None:
            project_root = os.getcwd()  # Use current working directory
        self.process_files(project_root, text_dir)

# Usage:
processor = CourtProcessor()
processor.run("./data/file_putusan/file_putusan")

Loaded 16572 training document IDs
Loaded 6666 test document IDs
Processing 23238 files (out of 23675 total) that match train/test IDs


Processing documents: 100%|██████████| 23238/23238 [21:47<00:00, 17.78file/s] 


Filtered out 1 records with lama hukuman = 88888
Saved standard training data to /home/jovyan/work/preprocessed_data/train_standard.csv
Standard training data shape: (16178, 3)
Saved standard test data to /home/jovyan/work/preprocessed_data/test_standard.csv
Standard test data shape: (6526, 2)
Filtered out 2 records with empty text
Saved different training data to /home/jovyan/work/preprocessed_data/train_anomaly.csv
Different training data shape: (350, 3)
Saved different test data to /home/jovyan/work/preprocessed_data/test_anomaly.csv
Different test data shape: (131, 2)
Saved training lifetime verdicts to /home/jovyan/work/preprocessed_data/train_lifetime.csv
Training lifetime verdicts shape: (41, 3)
Saved test lifetime verdicts to /home/jovyan/work/preprocessed_data/test_lifetime.csv
Test lifetime verdicts shape: (9, 2)

Summary:
Total files processed: 23188
Standard convention files: 22705 (97.9%)
Different convention files: 483 (2.1%)
Lifetime verdicts found: 50 (0.2%)
Dataset bre

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Modelling <a name="3"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Standard

In [ ]:
## ====================================================
## CONFIGURATION
## ====================================================
class Configs:
    # --- Environment & Paths ---
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    OUTPUT_DIR = Path("./results_baseline")
    BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

    # --- Preprocessed Data Paths ---
    PREPROCESSED_DIR = Path("./preprocessed_data")
    PREPROCESSED_TRAIN_PATH = PREPROCESSED_DIR / "train_standard.csv"
    PREPROCESSED_TEST_PATH = PREPROCESSED_DIR / "test_standard.csv"

    # --- Model & Training ---
    MODEL_NAME = (
        "indobenchmark/indobert-large-p2"  # Using baseline model without modifications
    )
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    EPOCHS = 30
    LEARNING_RATE = 2e-5
    SEED = 42

    # --- Training Arguments ---
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 16
    LR_SCHEDULER_TYPE = "linear"  # Using standard linear scheduler
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01


# --- Set Seed for Reproducibility ---
random.seed(Configs.SEED)
np.random.seed(Configs.SEED)
torch.manual_seed(Configs.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Configs.SEED)

# --- Ensure output directories exist ---
Configs.OUTPUT_DIR.mkdir(exist_ok=True)
Configs.BEST_MODEL_DIR.mkdir(exist_ok=True)
Configs.PREPROCESSED_DIR.mkdir(exist_ok=True)

print(f"Device: {Configs.DEVICE}")
print(f"Model: {Configs.MODEL_NAME}")
print(f"Output directory: {Configs.OUTPUT_DIR}")

# %%
## ====================================================
## LOAD AND PREPROCESS DATA
## ====================================================
print("=" * 50)
print("LOADING AND PREPROCESSING DATA")
print("=" * 50)

# Check if preprocessed data already exists
if Configs.PREPROCESSED_TRAIN_PATH.exists() and Configs.PREPROCESSED_TEST_PATH.exists():
    print("Loading existing preprocessed data...")
    train_df = pd.read_csv(Configs.PREPROCESSED_TRAIN_PATH)
    test_df = pd.read_csv(Configs.PREPROCESSED_TEST_PATH)
    print(f"Loaded preprocessed train data: {len(train_df)} samples")
    print(f"Loaded preprocessed test data: {len(test_df)} samples")

# Get target column name
target_col = "lama hukuman (bulan)"
print(f"\nTarget column: {target_col}")

# Filter training data (remove empty texts and invalid labels)
train_df = train_df[train_df["text"].str.len() > 0].reset_index(drop=True)
train_df = train_df[train_df[target_col] != 88888].reset_index(drop=True)

# Track empty test texts for later imputation
test_empty_mask = (test_df["text"].str.len() == 0).values

print(f"\nFinal train size after filtering: {len(train_df)}")
print(f"Test samples with empty text: {test_empty_mask.sum()}")

# Display sample of the data
print("\nSample of training data:")
print(train_df[["id", target_col]].head())
print(f"\nTarget statistics:")
print(train_df[target_col].describe())


# %%
## ====================================================
## DATASET CLASS FOR BASELINE MODEL
## ====================================================
class RegressionDataset(Dataset):
    """Dataset class for regression task"""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


# %%
## ====================================================
## METRICS COMPUTATION
## ====================================================
def compute_metrics(p):
    """Compute RMSE and MAE metrics"""
    preds = p.predictions.flatten()
    labels = p.label_ids.flatten()
    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)
    return {"rmse": rmse, "mae": mae}


# %%
## ====================================================
## PREPARE FULL TRAINING DATA
## ====================================================
print("\n" + "=" * 50)
print("PREPARING FULL TRAINING DATA")
print("=" * 50)

# Use full training data without validation split
X_train = train_df["text"].values
y_train = train_df[target_col].astype(float).values

print(f"Training samples: {len(X_train)}")
print(f"Training target mean: {y_train.mean():.2f}")

# %%
## ====================================================
## INITIALIZE TOKENIZER AND DATASETS
## ====================================================
print("\n" + "=" * 50)
print("INITIALIZING TOKENIZER AND DATASETS")
print("=" * 50)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(Configs.MODEL_NAME)
print(f"Tokenizer loaded: {Configs.MODEL_NAME}")

# Create datasets
train_dataset = RegressionDataset(X_train, y_train, tokenizer, Configs.MAX_LENGTH)
test_dataset = RegressionDataset(
    test_df["text"].values, [0] * len(test_df), tokenizer, Configs.MAX_LENGTH
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# %%
## ====================================================
## INITIALIZE BASELINE MODEL (NO MODIFICATIONS)
## ====================================================
print("\n" + "=" * 50)
print("INITIALIZING BASELINE MODEL")
print("=" * 50)

# Load the baseline IndoBERT model for sequence classification (regression)
# We set num_labels=1 for regression task
model = AutoModelForSequenceClassification.from_pretrained(
    Configs.MODEL_NAME,
    num_labels=1,  # Single output for regression
    problem_type="regression",  # Specify regression task
)

# Move model to device
model = model.to(Configs.DEVICE)

# Print model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {type(model).__name__}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {Configs.DEVICE}")

# %%
## ====================================================
## TRAINING ARGUMENTS
## ====================================================
print("\n" + "=" * 50)
print("SETTING UP TRAINING")
print("=" * 50)

training_args = TrainingArguments(
    output_dir=str(Configs.OUTPUT_DIR),
    num_train_epochs=Configs.EPOCHS,
    per_device_train_batch_size=Configs.BATCH_SIZE,
    gradient_accumulation_steps=Configs.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=Configs.LEARNING_RATE,
    weight_decay=Configs.WEIGHT_DECAY,
    lr_scheduler_type=Configs.LR_SCHEDULER_TYPE,
    warmup_ratio=Configs.WARMUP_RATIO,
    logging_strategy="steps",
    logging_steps=1000,  # Log every 100 steps
    save_strategy="epoch",  # Save at end of each epoch
    save_total_limit=2,  # Keep only last 2 checkpoints
    fp16=True if Configs.DEVICE == "cuda" else False,
    report_to="none",
    seed=Configs.SEED,
)

print(f"Epochs: {Configs.EPOCHS}")
print(f"Batch size: {Configs.BATCH_SIZE}")
print(f"Gradient accumulation steps: {Configs.GRADIENT_ACCUMULATION_STEPS}")
print(
    f"Effective batch size: {Configs.BATCH_SIZE * Configs.GRADIENT_ACCUMULATION_STEPS}"
)
print(f"Learning rate: {Configs.LEARNING_RATE}")
print(f"Warmup ratio: {Configs.WARMUP_RATIO}")
print(f"Weight decay: {Configs.WEIGHT_DECAY}")

# %%
## ====================================================
## INITIALIZE TRAINER AND START TRAINING
## ====================================================
print("\n" + "=" * 50)
print("TRAINING MODEL")
print("=" * 50)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Start training
print("Starting training with baseline IndoBERT model...")
print("Checkpoints will be saved at the end of each epoch")
trainer.train()

print("\nTraining completed!")

# %%
## ====================================================
## SAVE BEST MODEL AND TOKENIZER
## ====================================================
print("\n" + "=" * 50)
print("SAVING BEST MODEL")
print("=" * 50)

# Save the best model and tokenizer
print(f"Saving best model to: {Configs.BEST_MODEL_DIR}")
trainer.save_model(str(Configs.BEST_MODEL_DIR))
tokenizer.save_pretrained(str(Configs.BEST_MODEL_DIR))
print("Model and tokenizer saved successfully!")

# %%
## ====================================================
## GENERATE PREDICTIONS ON TEST SET
## ====================================================
print("\n" + "=" * 50)
print("GENERATING TEST PREDICTIONS")
print("=" * 50)

# Generate predictions
print("Generating predictions on test set...")
test_predictions = trainer.predict(test_dataset)
test_preds = test_predictions.predictions.flatten()

# Round predictions and clip negative values
final_preds = np.rint(np.clip(test_preds, 0, None)).astype(int)

# Handle empty test texts with imputation
if test_empty_mask.sum() > 0:
    imputation_value = int(round(np.mean(y_train)))
    print(
        f"Imputing {test_empty_mask.sum()} empty test texts with value: {imputation_value}"
    )
    final_preds[test_empty_mask] = imputation_value

# %%
## ====================================================
## CREATE AND SAVE SUBMISSION FILE
## ====================================================
print("\n" + "=" * 50)
print("CREATING SUBMISSION FILE")
print("=" * 50)

# Create submission dataframe
submission_df = pd.DataFrame({"id": test_df["id"], target_col: final_preds})

# Save submission file
submission_path = "submission_baseline.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved: {submission_path}")

# Display submission statistics
print(f"\nSubmission statistics:")
print(submission_df[target_col].describe())
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

# %%
## ====================================================
## MODEL TRAINING SUMMARY
## ====================================================
print("\n" + "=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)

print("Training completed using full dataset without validation split")
print(f"Total training samples used: {len(train_dataset)}")

Device: cuda
Model: indobenchmark/indobert-large-p2
Output directory: results_baseline
LOADING AND PREPROCESSING DATA
Loading existing preprocessed data...
Loaded preprocessed train data: 16178 samples
Loaded preprocessed test data: 6526 samples

Target column: lama hukuman (bulan)

Final train size after filtering: 16178
Test samples with empty text: 0

Sample of training data:
          id  lama hukuman (bulan)
0   doc_6989                    60
1  doc_10445                    54
2  doc_19228                    48
3  doc_15302                     3
4   doc_8170                     3

Target statistics:
count    16178.000000
mean        40.101619
std         37.628981
min          1.000000
25%         12.000000
50%         24.000000
75%         60.000000
max        240.000000
Name: lama hukuman (bulan), dtype: float64

PREPARING FULL TRAINING DATA
Training samples: 16178
Training target mean: 40.10

INITIALIZING TOKENIZER AND DATASETS
Tokenizer loaded: indobenchmark/indobert-large-p2


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: BertForSequenceClassification
Total parameters: 335,142,913
Trainable parameters: 335,142,913
Device: cuda

SETTING UP TRAINING
Epochs: 30
Batch size: 8
Gradient accumulation steps: 2
Effective batch size: 16
Learning rate: 2e-05
Warmup ratio: 0.1
Weight decay: 0.01

TRAINING MODEL
Starting training with baseline IndoBERT model...
Checkpoints will be saved at the end of each epoch


/tmp/ipykernel_14695/2560358225.py:237: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
1000,2259.838000
2000,1659.484600
3000,1158.057300
4000,754.671200
5000,538.651400
6000,427.829200
7000,373.654100
8000,313.222900
9000,287.362600
10000,255.572900



Training completed!

SAVING BEST MODEL
Saving best model to: results_baseline/best_model
Model and tokenizer saved successfully!

GENERATING TEST PREDICTIONS
Generating predictions on test set...



CREATING SUBMISSION FILE
Submission file saved: submission_baseline.csv

Submission statistics:
count    6526.000000
mean       38.036929
std        34.049102
min         0.000000
25%        12.000000
50%        24.000000
75%        61.000000
max       139.000000
Name: lama hukuman (bulan), dtype: float64

First 10 predictions:
          id  lama hukuman (bulan)
0   doc_1528                    16
1   doc_1027                     3
2   doc_7797                    10
3  doc_20197                    29
4   doc_8612                    25
5  doc_12485                    11
6   doc_5690                    67
7  doc_22883                     4
8   doc_8199                    17
9  doc_22148                    60

TRAINING SUMMARY
Training completed using full dataset without validation split
Total training samples used: 16178


## Anomaly

In [ ]:
## ====================================================
## CONFIGURATION
## ====================================================
class Configs:
    # --- Environment & Paths ---
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    OUTPUT_DIR = Path("./results_baseline_anomaly")
    BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

    # --- Preprocessed Data Paths ---
    PREPROCESSED_DIR = Path("./preprocessed_data")
    PREPROCESSED_TRAIN_PATH = PREPROCESSED_DIR / "train_anomaly.csv"
    PREPROCESSED_TEST_PATH = PREPROCESSED_DIR / "test_anomaly.csv"

    # --- Model & Training ---
    MODEL_NAME = (
        "indobenchmark/indobert-large-p2"  # Using baseline model without modifications
    )
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    EPOCHS = 30
    LEARNING_RATE = 2e-5
    SEED = 42

    # --- Training Arguments ---
    GRADIENT_ACCUMULATION_STEPS = 2  # Effective batch size = 16
    LR_SCHEDULER_TYPE = "linear"  # Using standard linear scheduler
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01


# --- Set Seed for Reproducibility ---
random.seed(Configs.SEED)
np.random.seed(Configs.SEED)
torch.manual_seed(Configs.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Configs.SEED)

# --- Ensure output directories exist ---
Configs.OUTPUT_DIR.mkdir(exist_ok=True)
Configs.BEST_MODEL_DIR.mkdir(exist_ok=True)
Configs.PREPROCESSED_DIR.mkdir(exist_ok=True)

print(f"Device: {Configs.DEVICE}")
print(f"Model: {Configs.MODEL_NAME}")
print(f"Output directory: {Configs.OUTPUT_DIR}")

# %%
## ====================================================
## LOAD AND PREPROCESS DATA
## ====================================================
print("=" * 50)
print("LOADING AND PREPROCESSING DATA")
print("=" * 50)

# Check if preprocessed data already exists
if Configs.PREPROCESSED_TRAIN_PATH.exists() and Configs.PREPROCESSED_TEST_PATH.exists():
    print("Loading existing preprocessed data...")
    train_df = pd.read_csv(Configs.PREPROCESSED_TRAIN_PATH)
    test_df = pd.read_csv(Configs.PREPROCESSED_TEST_PATH)
    print(f"Loaded preprocessed train data: {len(train_df)} samples")
    print(f"Loaded preprocessed test data: {len(test_df)} samples")

# Get target column name
target_col = "lama hukuman (bulan)"
print(f"\nTarget column: {target_col}")

# Filter training data (remove empty texts and invalid labels)
train_df = train_df[train_df["text"].str.len() > 0].reset_index(drop=True)
train_df = train_df[train_df[target_col] != 88888].reset_index(drop=True)

# Track empty test texts for later imputation
test_empty_mask = (test_df["text"].str.len() == 0).values

print(f"\nFinal train size after filtering: {len(train_df)}")
print(f"Test samples with empty text: {test_empty_mask.sum()}")

# Display sample of the data
print("\nSample of training data:")
print(train_df[["id", target_col]].head())
print(f"\nTarget statistics:")
print(train_df[target_col].describe())


# %%
## ====================================================
## DATASET CLASS FOR BASELINE MODEL
## ====================================================
class RegressionDataset(Dataset):
    """Dataset class for regression task"""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


# %%
## ====================================================
## METRICS COMPUTATION
## ====================================================
def compute_metrics(p):
    """Compute RMSE and MAE metrics"""
    preds = p.predictions.flatten()
    labels = p.label_ids.flatten()
    rmse = np.sqrt(mean_squared_error(labels, preds))
    mae = mean_absolute_error(labels, preds)
    return {"rmse": rmse, "mae": mae}


# %%
## ====================================================
## PREPARE FULL TRAINING DATA
## ====================================================
print("\n" + "=" * 50)
print("PREPARING FULL TRAINING DATA")
print("=" * 50)

# Use full training data without validation split
X_train = train_df["text"].values
y_train = train_df[target_col].astype(float).values

print(f"Training samples: {len(X_train)}")
print(f"Training target mean: {y_train.mean():.2f}")

# %%
## ====================================================
## INITIALIZE TOKENIZER AND DATASETS
## ====================================================
print("\n" + "=" * 50)
print("INITIALIZING TOKENIZER AND DATASETS")
print("=" * 50)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(Configs.MODEL_NAME)
print(f"Tokenizer loaded: {Configs.MODEL_NAME}")

# Create datasets
train_dataset = RegressionDataset(X_train, y_train, tokenizer, Configs.MAX_LENGTH)
test_dataset = RegressionDataset(
    test_df["text"].values, [0] * len(test_df), tokenizer, Configs.MAX_LENGTH
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# %%
## ====================================================
## INITIALIZE BASELINE MODEL (NO MODIFICATIONS)
## ====================================================
print("\n" + "=" * 50)
print("INITIALIZING BASELINE MODEL")
print("=" * 50)

# Load the baseline IndoBERT model for sequence classification (regression)
# We set num_labels=1 for regression task
model = AutoModelForSequenceClassification.from_pretrained(
    Configs.MODEL_NAME,
    num_labels=1,  # Single output for regression
    problem_type="regression",  # Specify regression task
)

# Move model to device
model = model.to(Configs.DEVICE)

# Print model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {type(model).__name__}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {Configs.DEVICE}")

# %%
## ====================================================
## TRAINING ARGUMENTS
## ====================================================
print("\n" + "=" * 50)
print("SETTING UP TRAINING")
print("=" * 50)

training_args = TrainingArguments(
    output_dir=str(Configs.OUTPUT_DIR),
    num_train_epochs=Configs.EPOCHS,
    per_device_train_batch_size=Configs.BATCH_SIZE,
    gradient_accumulation_steps=Configs.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=Configs.LEARNING_RATE,
    weight_decay=Configs.WEIGHT_DECAY,
    lr_scheduler_type=Configs.LR_SCHEDULER_TYPE,
    warmup_ratio=Configs.WARMUP_RATIO,
    logging_strategy="steps",
    logging_steps=500,  # Log every 100 steps
    save_strategy="epoch",  # Save at end of each epoch
    save_total_limit=2,  # Keep only last 2 checkpoints
    fp16=True if Configs.DEVICE == "cuda" else False,
    report_to="none",
    seed=Configs.SEED,
)

print(f"Epochs: {Configs.EPOCHS}")
print(f"Batch size: {Configs.BATCH_SIZE}")
print(f"Gradient accumulation steps: {Configs.GRADIENT_ACCUMULATION_STEPS}")
print(
    f"Effective batch size: {Configs.BATCH_SIZE * Configs.GRADIENT_ACCUMULATION_STEPS}"
)
print(f"Learning rate: {Configs.LEARNING_RATE}")
print(f"Warmup ratio: {Configs.WARMUP_RATIO}")
print(f"Weight decay: {Configs.WEIGHT_DECAY}")

# %%
## ====================================================
## INITIALIZE TRAINER AND START TRAINING
## ====================================================
print("\n" + "=" * 50)
print("TRAINING MODEL")
print("=" * 50)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

# Start training
print("Starting training with baseline IndoBERT model...")
print("Checkpoints will be saved at the end of each epoch")
trainer.train()

print("\nTraining completed!")

# %%
## ====================================================
## SAVE BEST MODEL AND TOKENIZER
## ====================================================
print("\n" + "=" * 50)
print("SAVING BEST MODEL")
print("=" * 50)

# Save the best model and tokenizer
print(f"Saving best model to: {Configs.BEST_MODEL_DIR}")
trainer.save_model(str(Configs.BEST_MODEL_DIR))
tokenizer.save_pretrained(str(Configs.BEST_MODEL_DIR))
print("Model and tokenizer saved successfully!")

# %%
## ====================================================
## GENERATE PREDICTIONS ON TEST SET
## ====================================================
print("\n" + "=" * 50)
print("GENERATING TEST PREDICTIONS")
print("=" * 50)

# Generate predictions
print("Generating predictions on test set...")
test_predictions = trainer.predict(test_dataset)
test_preds = test_predictions.predictions.flatten()

# Round predictions and clip negative values
final_preds = np.rint(np.clip(test_preds, 0, None)).astype(int)

# Handle empty test texts with imputation
if test_empty_mask.sum() > 0:
    imputation_value = int(round(np.mean(y_train)))
    print(
        f"Imputing {test_empty_mask.sum()} empty test texts with value: {imputation_value}"
    )
    final_preds[test_empty_mask] = imputation_value

# %%
## ====================================================
## CREATE AND SAVE SUBMISSION FILE
## ====================================================
print("\n" + "=" * 50)
print("CREATING SUBMISSION FILE")
print("=" * 50)

# Create submission dataframe
submission_df = pd.DataFrame({"id": test_df["id"], target_col: final_preds})

# Save submission file
submission_path = "submission_baseline.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission file saved: {submission_path}")

# Display submission statistics
print(f"\nSubmission statistics:")
print(submission_df[target_col].describe())
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

# %%
## ====================================================
## MODEL TRAINING SUMMARY
## ====================================================
print("\n" + "=" * 50)
print("TRAINING SUMMARY")
print("=" * 50)

print("Training completed using full dataset without validation split")
print(f"Total training samples used: {len(train_dataset)}")

Device: cuda
Model: indobenchmark/indobert-large-p2
Output directory: results_baseline_anomaly
LOADING AND PREPROCESSING DATA
Loading existing preprocessed data...
Loaded preprocessed train data: 350 samples
Loaded preprocessed test data: 131 samples

Target column: lama hukuman (bulan)

Final train size after filtering: 350
Test samples with empty text: 0

Sample of training data:
          id  lama hukuman (bulan)
0  doc_20062                     2
1  doc_18323                     2
2   doc_8245                     2
3  doc_21212                    60
4   doc_2738                     1

Target statistics:
count    350.000000
mean      12.874286
std       29.065506
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max      240.000000
Name: lama hukuman (bulan), dtype: float64

PREPARING FULL TRAINING DATA
Training samples: 350
Training target mean: 12.87

INITIALIZING TOKENIZER AND DATASETS
Tokenizer loaded: indobenchmark/indobert-large-p2
Train dataset s

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-large-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: BertForSequenceClassification
Total parameters: 335,142,913
Trainable parameters: 335,142,913
Device: cuda

SETTING UP TRAINING
Epochs: 30
Batch size: 8
Gradient accumulation steps: 2
Effective batch size: 16
Learning rate: 2e-05
Warmup ratio: 0.1
Weight decay: 0.01

TRAINING MODEL
Starting training with baseline IndoBERT model...
Checkpoints will be saved at the end of each epoch


/tmp/ipykernel_14695/3851890834.py:237: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,713.036900



Training completed!

SAVING BEST MODEL
Saving best model to: results_baseline_anomaly/best_model
Model and tokenizer saved successfully!

GENERATING TEST PREDICTIONS
Generating predictions on test set...



CREATING SUBMISSION FILE
Submission file saved: submission_baseline.csv

Submission statistics:
count    131.000000
mean       5.671756
std        7.676581
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       22.000000
Name: lama hukuman (bulan), dtype: float64

First 10 predictions:
          id  lama hukuman (bulan)
0   doc_5712                     2
1  doc_13881                     1
2  doc_22507                     2
3  doc_20139                     2
4  doc_21350                     1
5   doc_5506                     1
6  doc_11851                     2
7   doc_5972                    22
8   doc_6069                     1
9   doc_5054                     2

TRAINING SUMMARY
Training completed using full dataset without validation split
Total training samples used: 350


## GBDT Standard

In [ ]:
# %%
import os
import gc
import pickle
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    StackingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVR
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

# ML Libraries
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Tree-based models
from xgboost import XGBRegressor, train
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")


# %%
## ====================================================
## CONFIGURATION
## ====================================================
class Config:
    # Paths
    PREPROCESSED_DIR = Path("./preprocessed_data")
    PREPROCESSED_TRAIN_PATH = PREPROCESSED_DIR / "train_standard.csv"
    PREPROCESSED_TEST_PATH = PREPROCESSED_DIR / "test_standard.csv"

    # Embedding paths
    EMBEDDINGS_DIR = Path("./embeddings")
    TRAIN_EMBEDDINGS_PATH = EMBEDDINGS_DIR / "train_embeddings.npy"
    TEST_EMBEDDINGS_PATH = EMBEDDINGS_DIR / "test_embeddings.npy"
    TRAIN_LABELS_PATH = EMBEDDINGS_DIR / "train_labels.npy"

    # Model paths
    BERT_MODEL_PATH = Path("./results_baseline/best_model/")
    OUTPUT_DIR = Path("./gbdt_results")
    SUBMISSION_DIR = Path("./submissions")

    # Settings
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MAX_LENGTH = 512
    BATCH_SIZE = 32
    SEED = 42


# Create directories
Config.EMBEDDINGS_DIR.mkdir(exist_ok=True)
Config.OUTPUT_DIR.mkdir(exist_ok=True)

# Set seeds
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.SEED)

print(f"Device: {Config.DEVICE}")

# %%
## ====================================================
## STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS
## ====================================================
print("=" * 60)
print("STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS")
print("=" * 60)


class EmbeddingExtractor:
    def __init__(self, model_path, device="cuda"):
        self.device = device
        print(f"Loading model from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path)
        self.model = self.model.to(device)
        self.model.eval()
        print("Model loaded successfully!")

    def extract_embeddings(self, texts, batch_size=32):
        """Extract embeddings using [CLS] token"""
        embeddings = []

        with torch.no_grad():
            for i in tqdm(
                range(0, len(texts), batch_size), desc="Extracting embeddings"
            ):
                batch_texts = texts[i : i + batch_size]

                inputs = self.tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=Config.MAX_LENGTH,
                    return_tensors="pt",
                ).to(self.device)

                outputs = self.model(**inputs)
                # Use [CLS] token embedding
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(batch_embeddings)

                # Clear memory
                del inputs, outputs
                if i % (batch_size * 10) == 0:
                    torch.cuda.empty_cache()

        return np.vstack(embeddings)


# %%
## ====================================================
## STEP 2: EXTRACT AND SAVE EMBEDDINGS
## ====================================================
print("\n" + "=" * 60)
print("STEP 2: EXTRACT AND SAVE EMBEDDINGS")
print("=" * 60)

# Load data
train_df = pd.read_csv(Config.PREPROCESSED_TRAIN_PATH)
test_df = pd.read_csv(Config.PREPROCESSED_TEST_PATH)

# Get target column
target_col = next(
    c for c in train_df.columns if "bulan" in c.lower() and "lama" in c.lower()
)

# Filter valid training data
train_df = train_df[
    train_df["text"].notna() & (train_df["text"].str.len() > 0)
].reset_index(drop=True)
train_df = train_df[train_df[target_col] != 88888].reset_index(drop=True)

# Handle empty test texts
test_empty_mask = test_df["text"].isna() | (test_df["text"].str.len() == 0)
test_df.loc[test_empty_mask, "text"] = "dokumen kosong"

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Empty test samples: {test_empty_mask.sum()}")

# Extract or load embeddings
if (
    Config.TRAIN_EMBEDDINGS_PATH.exists()
    and Config.TEST_EMBEDDINGS_PATH.exists()
    and Config.TRAIN_LABELS_PATH.exists()
):
    print("\nLoading existing embeddings...")
    X_train_full = np.load(Config.TRAIN_EMBEDDINGS_PATH)
    X_test = np.load(Config.TEST_EMBEDDINGS_PATH)
    y_train_full = np.load(Config.TRAIN_LABELS_PATH)
    print(f"Loaded embeddings - Train: {X_train_full.shape}, Test: {X_test.shape}")

else:
    print("\nExtracting new embeddings...")

    # Initialize extractor
    extractor = EmbeddingExtractor(Config.BERT_MODEL_PATH, Config.DEVICE)

    # Extract embeddings
    print("\nExtracting training embeddings...")
    X_train_full = extractor.extract_embeddings(
        train_df["text"].tolist(), batch_size=Config.BATCH_SIZE
    )

    print("\nExtracting test embeddings...")
    X_test = extractor.extract_embeddings(
        test_df["text"].tolist(), batch_size=Config.BATCH_SIZE
    )

    # Get labels
    y_train_full = train_df[target_col].values.astype(float)

    # Save embeddings
    print("\nSaving embeddings...")
    np.save(Config.TRAIN_EMBEDDINGS_PATH, X_train_full)
    np.save(Config.TEST_EMBEDDINGS_PATH, X_test)
    np.save(Config.TRAIN_LABELS_PATH, y_train_full)
    print("Embeddings saved successfully!")

    # Clear memory
    del extractor
    torch.cuda.empty_cache()
    gc.collect()

# Use full training data without validation split
X_train = X_train_full
y_train = y_train_full

print(f"\nFull training data shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

# %%
## ====================================================
## STEP 3: MODEL TRAINING AND EVALUATION
## ====================================================
print("\n" + "=" * 60)
print("STEP 3: MODEL TRAINING ON FULL DATA")
print("=" * 60)

from sklearn.ensemble import HistGradientBoostingRegressor

models = {
    "hgb": HistGradientBoostingRegressor(
        random_state=Config.SEED,
    ),
    "xgb": XGBRegressor(random_state=Config.SEED, n_jobs=-1, tree_method="hist"),
    "lgbm": LGBMRegressor(
        **{
            "n_estimators": 1966,
            "max_depth": 6,
            "learning_rate": 0.023094911287456332,
            "colsample_bytree": 0.6391459395227008,
            "reg_alpha": 8.30987508958704,
            "reg_lambda": 2.207781368774698,
            "verbose": -1,
            "random_state": 42,
        }
    ),
    "catboost": CatBoostRegressor(
        random_state=Config.SEED,
        verbose=500,
        task_type="GPU" if Config.DEVICE == "cuda" else "CPU",
    ),
}

# Train each model on full data
results = {}

for name, model in models.items():
    print(f"\nTraining {name.upper()} on full dataset...")
    model.fit(X_train, y_train)

    # Evaluate on training data
    train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)

    print(f"{name.upper()} Results:")
    print(f"  Train RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}")

    results[name] = {"model": model, "train_rmse": train_rmse, "train_mae": train_mae}

# Save evaluation results
metrics_df = pd.DataFrame(
    {
        name: {"train_rmse": res["train_rmse"], "train_mae": res["train_mae"]}
        for name, res in results.items()
    }
).T

metrics_df.to_csv(Config.OUTPUT_DIR / "evaluation_metrics.csv")
print(f"\nMetrics saved to {Config.OUTPUT_DIR / 'evaluation_metrics.csv'}")

# %%
## ====================================================
## STEP 4: MAKE SUBMISSIONS
## ====================================================
print("\n" + "=" * 60)
print("STEP 4: MAKE SUBMISSIONS")
print("=" * 60)

# Imputation value for empty texts
imputation_value = int(round(np.mean(y_train)))
print(f"Imputation value for empty texts: {imputation_value}")

# Generate predictions for each model
for name, res in results.items():
    model = res["model"]

    # Predict
    predictions = model.predict(X_test)

    # Round and clip negative values
    predictions = np.rint(np.clip(predictions, 1, None)).astype(int)

    # Impute empty texts
    predictions[test_empty_mask] = imputation_value

    # Create submission
    submission = pd.DataFrame({"id": test_df["id"], target_col: predictions})

    # Save submission
    submission_path = Config.SUBMISSION_DIR / f"submission_{name}_standard.csv"
    submission.to_csv(submission_path, index=False)

    print(f"\n{name.upper()} submission:")
    print(f"  Saved to: {submission_path}")
    print(f"  Mean: {predictions.mean():.2f}, Std: {predictions.std():.2f}")
    print(f"  Min: {predictions.min()}, Max: {predictions.max()}")

# %%
## ====================================================
## SUMMARY
## ====================================================
print("\n" + "=" * 60)
print("PROCESS COMPLETED")
print("=" * 60)

print("\nSteps completed:")
print("1. ✓ Model loaded from:", Config.BERT_MODEL_PATH)
print("2. ✓ Embeddings extracted and saved to:", Config.EMBEDDINGS_DIR)
print("3. ✓ Models trained on full dataset (XGBoost, LightGBM, CatBoost)")
print("4. ✓ Submissions created in:", Config.OUTPUT_DIR)

print("\nFinal Results:")
print(metrics_df)

print("\nAll submissions saved:")
for name in results.keys():
    print(f"  - submission_{name}.csv")

# %%
## ====================================================
## HOW TO USE
## ====================================================
print("\n" + "=" * 60)
print("HOW TO USE THIS PIPELINE")
print("=" * 60)

usage = """
To run this pipeline:

1. Ensure you have the trained IndoBERT model in:
   ./results_baseline/best_model/

2. Ensure you have preprocessed data in:
   ./preprocessed_data/train_cleaned.csv
   ./preprocessed_data/test_cleaned.csv

3. Run this script:
   python gbdt_embedding_training.py

4. Results will be saved in:
   ./embeddings/          - Extracted embeddings
   ./gbdt_results/        - Submissions and metrics

To use saved embeddings for new experiments:
   - Embeddings will be automatically loaded if they exist
   - Delete the .npy files to re-extract embeddings

Output files:
   - submission_xgb.csv      - XGBoost predictions
   - submission_lgbm.csv     - LightGBM predictions
   - submission_catboost.csv - CatBoost predictions
   - evaluation_metrics.csv  - Model performance metrics
"""

print(usage)

Device: cuda
STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS

STEP 2: EXTRACT AND SAVE EMBEDDINGS
Train samples: 16178
Test samples: 6526
Empty test samples: 0

Extracting new embeddings...
Loading model from results_baseline/best_model
Model loaded successfully!

Extracting training embeddings...


Extracting embeddings: 100%|██████████| 506/506 [02:50<00:00,  2.96it/s]



Extracting test embeddings...


Extracting embeddings: 100%|██████████| 204/204 [01:05<00:00,  3.09it/s]



Saving embeddings...
Embeddings saved successfully!

Full training data shape: (16178, 1024)
Test shape: (6526, 1024)

STEP 3: MODEL TRAINING ON FULL DATA

Training HGB on full dataset...
HGB Results:
  Train RMSE: 2.9220, MAE: 1.0736

Training XGB on full dataset...
XGB Results:
  Train RMSE: 0.7284, MAE: 0.5061

Training LGBM on full dataset...
LGBM Results:
  Train RMSE: 0.9091, MAE: 0.5571

Training CATBOOST on full dataset...
Learning rate set to 0.058168
0:	learn: 35.5542659	total: 46.2ms	remaining: 46.2s
500:	learn: 3.0356328	total: 17s	remaining: 16.9s
999:	learn: 2.3592778	total: 32.6s	remaining: 0us
CATBOOST Results:
  Train RMSE: 2.3593, MAE: 1.2640

Metrics saved to gbdt_results/evaluation_metrics.csv

STEP 4: MAKE SUBMISSIONS
Imputation value for empty texts: 40

HGB submission:
  Saved to: submissions/submission_hgb_standard.csv
  Mean: 38.60, Std: 34.94
  Min: 1, Max: 215

XGB submission:
  Saved to: submissions/submission_xgb_standard.csv
  Mean: 38.53, Std: 34.96
  Mi

## GBDT Anomaly

In [ ]:
# %%
import os
import gc
import pickle
import warnings
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    StackingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.svm import SVR
from tqdm import tqdm

import torch
from transformers import AutoTokenizer, AutoModel

# ML Libraries
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Tree-based models
from xgboost import XGBRegressor, train
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")


# %%
## ====================================================
## CONFIGURATION
## ====================================================
class Config:
    # Paths
    PREPROCESSED_DIR = Path("./preprocessed_data")
    PREPROCESSED_TRAIN_PATH = PREPROCESSED_DIR / "train_anomaly.csv"
    PREPROCESSED_TEST_PATH = PREPROCESSED_DIR / "test_anomaly.csv"

    # Embedding paths
    EMBEDDINGS_DIR = Path("./embeddings_anomaly")
    TRAIN_EMBEDDINGS_PATH = EMBEDDINGS_DIR / "train_embeddings.npy"
    TEST_EMBEDDINGS_PATH = EMBEDDINGS_DIR / "test_embeddings.npy"
    TRAIN_LABELS_PATH = EMBEDDINGS_DIR / "train_labels.npy"

    # Model paths
    BERT_MODEL_PATH = Path("./results_baseline_anomaly/best_model/")
    OUTPUT_DIR = Path("./gbdt_results_anomaly")
    SUBMISSION_DIR = Path("./submissions")

    # Settings
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MAX_LENGTH = 512
    BATCH_SIZE = 32
    SEED = 42


# Create directories
Config.EMBEDDINGS_DIR.mkdir(exist_ok=True)
Config.OUTPUT_DIR.mkdir(exist_ok=True)

# Set seeds
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Config.SEED)

print(f"Device: {Config.DEVICE}")

# %%
## ====================================================
## STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS
## ====================================================
print("=" * 60)
print("STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS")
print("=" * 60)


class EmbeddingExtractor:
    def __init__(self, model_path, device="cuda"):
        self.device = device
        print(f"Loading model from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path)
        self.model = self.model.to(device)
        self.model.eval()
        print("Model loaded successfully!")

    def extract_embeddings(self, texts, batch_size=32):
        """Extract embeddings using [CLS] token"""
        embeddings = []

        with torch.no_grad():
            for i in tqdm(
                range(0, len(texts), batch_size), desc="Extracting embeddings"
            ):
                batch_texts = texts[i : i + batch_size]

                inputs = self.tokenizer(
                    batch_texts,
                    padding=True,
                    truncation=True,
                    max_length=Config.MAX_LENGTH,
                    return_tensors="pt",
                ).to(self.device)

                outputs = self.model(**inputs)
                # Use [CLS] token embedding
                batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                embeddings.append(batch_embeddings)

                # Clear memory
                del inputs, outputs
                if i % (batch_size * 10) == 0:
                    torch.cuda.empty_cache()

        return np.vstack(embeddings)


# %%
## ====================================================
## STEP 2: EXTRACT AND SAVE EMBEDDINGS
## ====================================================
print("\n" + "=" * 60)
print("STEP 2: EXTRACT AND SAVE EMBEDDINGS")
print("=" * 60)

# Load data
train_df = pd.read_csv(Config.PREPROCESSED_TRAIN_PATH)
test_df = pd.read_csv(Config.PREPROCESSED_TEST_PATH)

# Get target column
target_col = next(
    c for c in train_df.columns if "bulan" in c.lower() and "lama" in c.lower()
)

# Filter valid training data
train_df = train_df[
    train_df["text"].notna() & (train_df["text"].str.len() > 0)
].reset_index(drop=True)
train_df = train_df[train_df[target_col] != 88888].reset_index(drop=True)

# Handle empty test texts
test_empty_mask = test_df["text"].isna() | (test_df["text"].str.len() == 0)
test_df.loc[test_empty_mask, "text"] = "dokumen kosong"

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Empty test samples: {test_empty_mask.sum()}")

# Extract or load embeddings
if (
    Config.TRAIN_EMBEDDINGS_PATH.exists()
    and Config.TEST_EMBEDDINGS_PATH.exists()
    and Config.TRAIN_LABELS_PATH.exists()
):
    print("\nLoading existing embeddings...")
    X_train_full = np.load(Config.TRAIN_EMBEDDINGS_PATH)
    X_test = np.load(Config.TEST_EMBEDDINGS_PATH)
    y_train_full = np.load(Config.TRAIN_LABELS_PATH)
    print(f"Loaded embeddings - Train: {X_train_full.shape}, Test: {X_test.shape}")

else:
    print("\nExtracting new embeddings...")

    # Initialize extractor
    extractor = EmbeddingExtractor(Config.BERT_MODEL_PATH, Config.DEVICE)

    # Extract embeddings
    print("\nExtracting training embeddings...")
    X_train_full = extractor.extract_embeddings(
        train_df["text"].tolist(), batch_size=Config.BATCH_SIZE
    )

    print("\nExtracting test embeddings...")
    X_test = extractor.extract_embeddings(
        test_df["text"].tolist(), batch_size=Config.BATCH_SIZE
    )

    # Get labels
    y_train_full = train_df[target_col].values.astype(float)

    # Save embeddings
    print("\nSaving embeddings...")
    np.save(Config.TRAIN_EMBEDDINGS_PATH, X_train_full)
    np.save(Config.TEST_EMBEDDINGS_PATH, X_test)
    np.save(Config.TRAIN_LABELS_PATH, y_train_full)
    print("Embeddings saved successfully!")

    # Clear memory
    del extractor
    torch.cuda.empty_cache()
    gc.collect()

# Use full training data without validation split
X_train = X_train_full
y_train = y_train_full

print(f"\nFull training data shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")

# %%
## ====================================================
## STEP 3: MODEL TRAINING AND EVALUATION
## ====================================================
print("\n" + "=" * 60)
print("STEP 3: MODEL TRAINING ON FULL DATA")
print("=" * 60)

from sklearn.ensemble import HistGradientBoostingRegressor

models = {
    "hgb": HistGradientBoostingRegressor(
        random_state=Config.SEED,
    ),
    "xgb": XGBRegressor(random_state=Config.SEED, n_jobs=-1, tree_method="hist"),
    "lgbm": LGBMRegressor(
        **{
            "n_estimators": 1966,
            "max_depth": 6,
            "learning_rate": 0.023094911287456332,
            "colsample_bytree": 0.6391459395227008,
            "reg_alpha": 8.30987508958704,
            "reg_lambda": 2.207781368774698,
            "verbose": -1,
            "random_state": 42,
        }
    ),
    "catboost": CatBoostRegressor(
        random_state=Config.SEED,
        verbose=500,
        task_type="GPU" if Config.DEVICE == "cuda" else "CPU",
    ),
}

# Train each model on full data
results = {}

for name, model in models.items():
    print(f"\nTraining {name.upper()} on full dataset...")
    model.fit(X_train, y_train)

    # Evaluate on training data
    train_pred = model.predict(X_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    train_mae = mean_absolute_error(y_train, train_pred)

    print(f"{name.upper()} Results:")
    print(f"  Train RMSE: {train_rmse:.4f}, MAE: {train_mae:.4f}")

    results[name] = {"model": model, "train_rmse": train_rmse, "train_mae": train_mae}

# Save evaluation results
metrics_df = pd.DataFrame(
    {
        name: {"train_rmse": res["train_rmse"], "train_mae": res["train_mae"]}
        for name, res in results.items()
    }
).T

metrics_df.to_csv(Config.OUTPUT_DIR / "evaluation_metrics.csv")
print(f"\nMetrics saved to {Config.OUTPUT_DIR / 'evaluation_metrics.csv'}")

# %%
## ====================================================
## STEP 4: MAKE SUBMISSIONS
## ====================================================
print("\n" + "=" * 60)
print("STEP 4: MAKE SUBMISSIONS")
print("=" * 60)

# Imputation value for empty texts
imputation_value = int(round(np.mean(y_train)))
print(f"Imputation value for empty texts: {imputation_value}")

# Generate predictions for each model
for name, res in results.items():
    model = res["model"]

    # Predict
    predictions = model.predict(X_test)

    # Round and clip negative values
    predictions = np.rint(np.clip(predictions, 1, None)).astype(int)

    # Impute empty texts
    predictions[test_empty_mask] = imputation_value

    # Create submission
    submission = pd.DataFrame({"id": test_df["id"], target_col: predictions})

    # Save submission
    submission_path = Config.SUBMISSION_DIR / f"submission_{name}_anomaly.csv"
    submission.to_csv(submission_path, index=False)

    print(f"\n{name.upper()} submission:")
    print(f"  Saved to: {submission_path}")
    print(f"  Mean: {predictions.mean():.2f}, Std: {predictions.std():.2f}")
    print(f"  Min: {predictions.min()}, Max: {predictions.max()}")

# %%
## ====================================================
## SUMMARY
## ====================================================
print("\n" + "=" * 60)
print("PROCESS COMPLETED")
print("=" * 60)

print("\nSteps completed:")
print("1. ✓ Model loaded from:", Config.BERT_MODEL_PATH)
print("2. ✓ Embeddings extracted and saved to:", Config.EMBEDDINGS_DIR)
print("3. ✓ Models trained on full dataset (XGBoost, LightGBM, CatBoost)")
print("4. ✓ Submissions created in:", Config.OUTPUT_DIR)

print("\nFinal Results:")
print(metrics_df)

print("\nAll submissions saved:")
for name in results.keys():
    print(f"  - submission_{name}.csv")

Device: cuda
STEP 1: LOAD MODEL AND EXTRACT EMBEDDINGS

STEP 2: EXTRACT AND SAVE EMBEDDINGS
Train samples: 350
Test samples: 131
Empty test samples: 0

Extracting new embeddings...
Loading model from results_baseline_anomaly/best_model
Model loaded successfully!

Extracting training embeddings...


Extracting embeddings: 100%|██████████| 11/11 [00:08<00:00,  1.35it/s]



Extracting test embeddings...


Extracting embeddings: 100%|██████████| 5/5 [00:03<00:00,  1.62it/s]



Saving embeddings...
Embeddings saved successfully!

Full training data shape: (350, 1024)
Test shape: (131, 1024)

STEP 3: MODEL TRAINING ON FULL DATA

Training HGB on full dataset...
HGB Results:
  Train RMSE: 6.6389, MAE: 1.2285

Training XGB on full dataset...
XGB Results:
  Train RMSE: 2.7754, MAE: 0.2860

Training LGBM on full dataset...
LGBM Results:
  Train RMSE: 3.0826, MAE: 0.6331

Training CATBOOST on full dataset...
Learning rate set to 0.035203
0:	learn: 28.3879533	total: 30.3ms	remaining: 30.2s
500:	learn: 3.8259646	total: 13.7s	remaining: 13.6s
999:	learn: 3.1519378	total: 27.2s	remaining: 0us
CATBOOST Results:
  Train RMSE: 3.1519, MAE: 0.9334

Metrics saved to gbdt_results_anomaly/evaluation_metrics.csv

STEP 4: MAKE SUBMISSIONS
Imputation value for empty texts: 13

HGB submission:
  Saved to: submissions/submission_hgb_anomaly.csv
  Mean: 11.28, Std: 23.07
  Min: 1, Max: 107

XGB submission:
  Saved to: submissions/submission_xgb_anomaly.csv
  Mean: 10.76, Std: 24.15

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Submission <a name="4"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

In [ ]:
# --- Hardcoded file paths ---
sample_path = Path("./data/sample_submission.csv")
pred1_path = Path("./submissions/submission_catboost_standard.csv")
pred2_path = Path("./submissions/submission_catboost_anomaly.csv")
test_standard_path = Path("./preprocessed_data/test_standard.csv")
out_path = Path("./submissions/submission_100_SAL_CAT.csv")

# --- Load sample submission ---
sample = pd.read_csv(sample_path)

# --- Load test standard data to check lifetime column ---
test_standard = pd.read_csv(test_standard_path)

# Load test lifetime data to get lifetime IDs
test_lifetime_path = Path("./preprocessed_data/test_lifetime.csv")
test_lifetime = pd.read_csv(test_lifetime_path)

# Load predictions
pred1 = pd.read_csv(pred1_path)
pred2 = pd.read_csv(pred2_path)

# Concatenate predictions
preds = pd.concat([pred1, pred2], ignore_index=True)

# Merge predictions into sample_submission by 'id'
merged = sample.drop('lama hukuman (bulan)', axis=1).merge(
    preds, on='id', how='left')

# If prediction column is not named 'lama hukuman (bulan)', rename it
if 'lama hukuman (bulan)' not in merged.columns:
    pred_col = [c for c in merged.columns if c != 'id'][0]
    merged = merged.rename(columns={pred_col: 'lama hukuman (bulan)'})

# Check for missing predictions and log them
missing_mask = merged['lama hukuman (bulan)'].isna()
missing_ids = merged[missing_mask]['id'].tolist()

if missing_ids:
    print(f"Warning: {len(missing_ids)} IDs missing predictions:")
    for missing_id in missing_ids:
        print(f"  {missing_id}")
else:
    print("All IDs have predictions!")

# Fill missing predictions with 0 (optional, or keep as is)
merged['lama hukuman (bulan)'] = merged['lama hukuman (bulan)'].fillna(0)

# Apply lifetime rule: set to 240 for IDs that exist in test_lifetime.csv
lifetime_ids = test_lifetime['id'].tolist()
lifetime_mask = merged['id'].isin(lifetime_ids)
lifetime_count = lifetime_mask.sum()

if lifetime_count > 0:
    # Capture original values before overwrite
    lifetime_original = merged.loc[lifetime_mask, ['id', 'lama hukuman (bulan)']].copy()
    merged.loc[lifetime_mask, 'lama hukuman (bulan)'] = 240
    print(f"Applied lifetime rule: {lifetime_count} IDs set to 240 months")
    print("Lifetime IDs updated (id: original -> new):")
    for _, row in lifetime_original.iterrows():
        orig_val = row['lama hukuman (bulan)']
        arrow = '->' if orig_val != 240 else '=='
        print(f"  {row['id']}: {orig_val} {arrow} 240")
else:
    print("No lifetime IDs found in the submission")

# Save to output
merged.to_csv(out_path, index=False)
print(f"Final submission saved to: {out_path}")

  doc_5707
  doc_3659
  doc_16131
  doc_3427
  doc_13481
  doc_2547
  doc_10141
  doc_21429
  doc_7723
Applied lifetime rule: 9 IDs set to 240 months
Lifetime IDs updated (id: original -> new):
  doc_5707: 0.0 -> 240
  doc_3659: 0.0 -> 240
  doc_16131: 0.0 -> 240
  doc_3427: 0.0 -> 240
  doc_13481: 0.0 -> 240
  doc_2547: 0.0 -> 240
  doc_10141: 0.0 -> 240
  doc_21429: 0.0 -> 240
  doc_7723: 0.0 -> 240
Final submission saved to: submissions/submission_100_SAL_CAT.csv


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Thanks

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">